# openoppsdb manager

This notebook is connected to `wyattowalsh/openoppsdb`. Schedule it with a Kaggle cron cadence such as `0 */6 * * *`. Each run installs OpenOpps from GitHub, copies the current dataset SQLite file, syncs public jobs, exports every table, regenerates metadata, and publishes a new dataset version.


In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
from datetime import UTC, datetime
import urllib.request

DATASET_ID = os.environ.get(
    "OPENOPPS_KAGGLE_DATASET",
    "wyattowalsh/openoppsdb",
)
PACKAGE_SPEC = os.environ.get(
    "OPENOPPS_PACKAGE_SPEC",
    "git+https://github.com/wyattowalsh/openopps.git@main",
)
OUTPUT_DIR = Path(
    os.environ.get(
        "OPENOPPS_KAGGLE_OUTPUT_DIR",
        "/kaggle/working/openoppsdb",
    )
)
DB_PATH = OUTPUT_DIR / "openopps.sqlite"
GENERATOR_SCRIPT = OUTPUT_DIR / "generate_kaggle_metadata.py"
CSV_DIR = "exports/csv"
PARQUET_DIR = "exports/parquet"
KAGGLE_INPUT_DIR = Path("/kaggle/input")
GENERATOR_SCRIPT_URL = os.environ.get(
    "OPENOPPS_GENERATOR_SCRIPT_URL",
    "https://raw.githubusercontent.com/wyattowalsh/openopps/main/scripts/generate_kaggle_metadata.py",
)
DATASET_IMAGE_URL = os.environ.get(
    "OPENOPPS_DATASET_IMAGE_URL",
    "https://raw.githubusercontent.com/wyattowalsh/openopps/main/docs/public/social/openoppsdb.png",
)

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def run(command: list[str], *, env: dict[str, str] | None = None) -> None:
    print("+", " ".join(command))
    subprocess.run(command, check=True, env=env)

db_candidates = sorted(KAGGLE_INPUT_DIR.glob("**/openopps.sqlite"))
if db_candidates:
    source_db = max(db_candidates, key=lambda path: path.stat().st_mtime)
    shutil.copy2(source_db, DB_PATH)
    print(f"Copied prior OpenOpps DB snapshot from {source_db} to {DB_PATH}")
else:
    print("No prior OpenOpps DB snapshot found; creating a new ledger.")

run([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", PACKAGE_SPEC, "kaggle"])
urllib.request.urlretrieve(GENERATOR_SCRIPT_URL, GENERATOR_SCRIPT)
urllib.request.urlretrieve(DATASET_IMAGE_URL, OUTPUT_DIR / "dataset-cover-image.png")


In [ ]:
openopps_env = os.environ.copy()
openopps_env["OPENOPPS_DB_URL"] = f"sqlite:///{DB_PATH}"
openopps_env["OPENOPPS_CACHE_ENABLED"] = "false"

run(["openopps", "admin", "db", "init"], env=openopps_env)
run(["openopps", "sync", "--metrics-json"], env=openopps_env)


In [ ]:
run([
    sys.executable,
    str(GENERATOR_SCRIPT),
    "--output-dir",
    str(OUTPUT_DIR),
    "--data-db",
    str(DB_PATH),
    "--manager-dir",
    str(OUTPUT_DIR / "_manager-unused"),
])
shutil.rmtree(OUTPUT_DIR / "_manager-unused", ignore_errors=True)

for path in sorted(OUTPUT_DIR.iterdir()):
    if path.name.endswith(".cache.db") or path.name == "generate_kaggle_metadata.py":
        path.unlink()
        continue
    print(path.name, path.stat().st_size)


In [ ]:
message = f"Scheduled OpenOpps active-job snapshot {datetime.now(UTC).isoformat()}"
kaggle_json = Path.home() / ".kaggle" / "kaggle.json"
token_path = os.environ.get("KAGGLE_API_V1_TOKEN_PATH")
has_kaggle_credentials = bool(
    os.environ.get("KAGGLE_API_TOKEN")
    or (token_path and Path(token_path).exists())
    or (os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"))
    or kaggle_json.exists()
)

if has_kaggle_credentials:
    run([
        "kaggle",
        "datasets",
        "version",
        "-p",
        str(OUTPUT_DIR),
        "-m",
        message,
        "-q",
        "-t",
        "-r",
        "zip",
    ])
else:
    print("Skipping dataset version upload because Kaggle API credentials are unavailable.")
